# CUSTOMER ESCALATION RISK ANALYSIS
## 06 — Final Findings & Management Actions

**Institution:** Dhanekula Institute of Engineering and Technology  
**Department:** Computer Science and Engineering  
**Team:** Team 3  
**Role:** Integration and Final Evidence Consolidation

---

### Purpose

Consolidate evidence from Problem Statements 1–7 and translate the strongest supported operational patterns into **exactly three prioritised management actions**.

### Analysis principles

- Distinguish observed results from interpretation and proposed benefits.
- Report counts together with percentages and denominators.
- Treat `Open` and `Pending Customer Response` as unresolved statuses, **not as confirmed escalations**.
- Do not calculate escalation rates without an explicit escalation indicator.
- Do not claim queue-level results because no queue field is available.
- Do not infer response or resolution durations from timestamp fields without a ticket-created timestamp.
- Do not include customer names, emails, or raw ticket descriptions in outputs.

> **Integration status:** This notebook independently reproduces the key evidence visible in the current team analysis. Member 4's corrected service-performance results should be reviewed and incorporated before the final recommendations are submitted.

## 1. Setup and reproducibility

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

DATA_PATH = Path("../data/raw/customer_support_tickets.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}\n"
        "Place the organiser-provided CSV at data/raw/customer_support_tickets.csv."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

## 2. Dataset validation

In [ ]:
required_columns = [
    "Ticket ID", "Ticket Type", "Ticket Status", "Ticket Priority",
    "Ticket Channel", "First Response Time", "Time to Resolution",
    "Customer Satisfaction Rating"
]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Required columns missing from dataset: {missing_columns}")

validation_summary = pd.DataFrame({
    "Measure": [
        "Total records",
        "Total columns",
        "Exact duplicate rows",
        "Duplicate Ticket IDs"
    ],
    "Result": [
        len(df),
        df.shape[1],
        int(df.duplicated().sum()),
        int(df["Ticket ID"].duplicated().sum())
    ]
})
display(validation_summary)

missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
}).sort_values("Missing Count", ascending=False)

display(missing_summary)

## 3. Overall ticket status and priority

In [ ]:
total_tickets = len(df)

status_summary = (
    df["Ticket Status"].fillna("Missing").value_counts()
    .rename_axis("Ticket Status")
    .reset_index(name="Ticket Count")
)
status_summary["Percentage"] = (
    status_summary["Ticket Count"] / total_tickets * 100
).round(2)
display(status_summary)

priority_summary = (
    df["Ticket Priority"].fillna("Missing").value_counts()
    .rename_axis("Ticket Priority")
    .reset_index(name="Ticket Count")
)
priority_summary["Percentage"] = (
    priority_summary["Ticket Count"] / total_tickets * 100
).round(2)
display(priority_summary)

unresolved_statuses = {"Open", "Pending Customer Response"}
unresolved_count = int(df["Ticket Status"].isin(unresolved_statuses).sum())
closed_count = int((df["Ticket Status"] == "Closed").sum())

print(f"Open + Pending Customer Response: {unresolved_count:,} "
      f"({unresolved_count / total_tickets * 100:.2f}%)")
print(f"Closed: {closed_count:,} ({closed_count / total_tickets * 100:.2f}%)")

## 4. Ticket type hotspot analysis

In [ ]:
def build_segment_summary(data, segment_column):
    result = (
        data.groupby(segment_column, dropna=False)
        .agg(
            Total_Tickets=("Ticket ID", "count"),
            Open_Tickets=("Ticket Status", lambda values: (values == "Open").sum()),
            Pending_Tickets=(
                "Ticket Status",
                lambda values: (values == "Pending Customer Response").sum()
            ),
            Closed_Tickets=("Ticket Status", lambda values: (values == "Closed").sum())
        )
    )
    result["Unresolved_Tickets"] = (
        result["Open_Tickets"] + result["Pending_Tickets"]
    )
    result["Unresolved_Percentage"] = (
        result["Unresolved_Tickets"] / result["Total_Tickets"] * 100
    ).round(2)
    result["Ticket_Share_Percentage"] = (
        result["Total_Tickets"] / len(data) * 100
    ).round(2)
    return result.sort_values("Unresolved_Percentage", ascending=False)

ticket_type_summary = build_segment_summary(df, "Ticket Type")
display(ticket_type_summary)

## 5. Support channel hotspot analysis

In [ ]:
channel_summary = build_segment_summary(df, "Ticket Channel")
display(channel_summary)

## 6. Visual evidence

In [ ]:
# Ticket type unresolved proportion
fig, ax = plt.subplots(figsize=(10, 5))
plot_data = ticket_type_summary["Unresolved_Percentage"].sort_values()
bars = ax.barh(plot_data.index.astype(str), plot_data.values)
ax.set_title("Unresolved Ticket Percentage by Ticket Type", weight="bold", fontsize=14)
ax.set_xlabel("Unresolved tickets (%)")
ax.set_ylabel("Ticket type")
ax.set_xlim(0, 85)
ax.grid(axis="x", linestyle="--", alpha=0.35)
ax.bar_label(bars, labels=[f"{v:.2f}%" for v in plot_data.values], padding=4)
plt.tight_layout()
plt.show()

# Support channel unresolved proportion
fig, ax = plt.subplots(figsize=(10, 5))
plot_data = channel_summary["Unresolved_Percentage"].sort_values()
bars = ax.barh(plot_data.index.astype(str), plot_data.values)
ax.set_title("Unresolved Ticket Percentage by Support Channel", weight="bold", fontsize=14)
ax.set_xlabel("Unresolved tickets (%)")
ax.set_ylabel("Support channel")
ax.set_xlim(0, 85)
ax.grid(axis="x", linestyle="--", alpha=0.35)
ax.bar_label(bars, labels=[f"{v:.2f}%" for v in plot_data.values], padding=4)
plt.tight_layout()
plt.show()

## 7. Customer satisfaction analysis

In [ ]:
csat = pd.to_numeric(df["Customer Satisfaction Rating"], errors="coerce")
rated_csat = csat.dropna()

csat_metrics = pd.DataFrame({
    "Metric": [
        "Tickets with recorded CSAT",
        "Tickets missing CSAT",
        "CSAT coverage (%)",
        "Average CSAT",
        "Median CSAT"
    ],
    "Result": [
        len(rated_csat),
        int(csat.isna().sum()),
        round(csat.notna().mean() * 100, 2),
        round(rated_csat.mean(), 2) if len(rated_csat) else np.nan,
        round(rated_csat.median(), 2) if len(rated_csat) else np.nan
    ]
})
display(csat_metrics)

csat_distribution = (
    rated_csat.value_counts().sort_index()
    .rename_axis("CSAT Rating")
    .reset_index(name="Ticket Count")
)
if len(rated_csat):
    csat_distribution["Percentage of Rated Tickets"] = (
        csat_distribution["Ticket Count"] / len(rated_csat) * 100
    ).round(2)
display(csat_distribution)

if len(rated_csat):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bars = ax.bar(
        csat_distribution["CSAT Rating"].astype(str),
        csat_distribution["Ticket Count"]
    )
    ax.set_title("Customer Satisfaction Rating Distribution", weight="bold", fontsize=14)
    ax.set_xlabel("Customer satisfaction rating (1–5)")
    ax.set_ylabel("Number of rated tickets")
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.bar_label(bars, padding=3)
    plt.tight_layout()
    plt.show()

## 8. Service timestamp evidence and integration checkpoint

In [ ]:
# Parse timestamps as datetimes, not numeric values.
first_response_dt = pd.to_datetime(df["First Response Time"], errors="coerce")
resolution_dt = pd.to_datetime(df["Time to Resolution"], errors="coerce")

timestamp_coverage = pd.DataFrame({
    "Timestamp Field": ["First Response Time", "Time to Resolution"],
    "Available Records": [first_response_dt.notna().sum(), resolution_dt.notna().sum()],
    "Missing / Invalid": [first_response_dt.isna().sum(), resolution_dt.isna().sum()]
})
timestamp_coverage["Availability (%)"] = (
    timestamp_coverage["Available Records"] / len(df) * 100
).round(2)
display(timestamp_coverage)

paired = first_response_dt.notna() & resolution_dt.notna()
interval = resolution_dt[paired] - first_response_dt[paired]
negative_interval_count = int((interval < pd.Timedelta(0)).sum())
valid_interval = interval[interval >= pd.Timedelta(0)]

print("Records with both timestamps:", int(paired.sum()))
print("Negative timestamp intervals:", negative_interval_count)
print("Valid non-negative intervals:", len(valid_interval))

if len(valid_interval):
    interval_hours = valid_interval.dt.total_seconds() / 3600
    display(pd.DataFrame({
        "Measure": ["Median interval (hours)", "Mean interval (hours)", "P90 interval (hours)"],
        "Value": [
            interval_hours.median(),
            interval_hours.mean(),
            interval_hours.quantile(0.90)
        ]
    }).round(2))

print(
    "Definition: this is the interval between the recorded first-response "
    "and resolution timestamps only. It is NOT total resolution duration "
    "from ticket creation."
)

## 9. Consolidated evidence register

The tables generated above should be used as the reproducible source of truth. The following observations are supported by the current analysis outputs:

| Evidence area | Observed result | Interpretation boundary |
|---|---|---|
| Dataset size | 8,469 records and 17 columns | Describes this supplied dataset |
| Duplicate checks | 0 exact duplicate rows and 0 duplicate Ticket IDs | Does not prove all fields are error-free |
| Unresolved workload | 5,700 Open or Pending Customer Response tickets (67.30%) | Unresolved status is not escalation |
| Ticket type | Cancellation requests: 1,179 of 1,695 unresolved (69.56%) | Highest unresolved proportion among listed ticket types |
| Channel | Social media: 1,437 of 2,121 unresolved (67.75%) | Highest channel proportion, but channel differences are small |
| CSAT | 2,769 ratings; average 2.99/5; median 3 | Applies only to records with ratings |

### Service-performance integration

Run this notebook after Member 4's corrected timestamp analysis. Confirm that the timestamp coverage, negative interval count, and valid interval statistics agree. Do not copy figures manually if they differ; investigate and document the discrepancy.

## 10. Exactly three prioritised management actions — provisional

> **Status: Provisional.** These actions are grounded in the evidence currently available in this notebook. Review and adjust their priority after incorporating Member 4's corrected service-performance outputs. Do not describe the projected benefits as measured outcomes.

### Priority 1 — Review and streamline cancellation-request handling

**Observed risk pattern:** Cancellation requests have the highest unresolved proportion among the analysed ticket types.

**Evidence:** 1,179 of 1,695 cancellation requests are Open or Pending Customer Response (69.56%).

**Operational action:** Review the cancellation workflow from receipt to closure. Define clear ownership, document required customer information, establish a regular review of ageing open cancellation requests, and identify common reasons for pending status.

**Projected / hypothetical benefit:** A clearer workflow may improve visibility and consistency in cancellation handling and may reduce avoidable pending work. This benefit must be measured after implementation.

### Priority 2 — Review follow-up practices for unresolved tickets

**Observed risk pattern:** 5,700 of 8,469 tickets are classified as Open or Pending Customer Response (67.30%).

**Evidence:** 2,819 tickets are Open and 2,881 are Pending Customer Response.

**Operational action:** Introduce a consistent follow-up review process for open and pending tickets. Assign ownership, define review intervals, and record follow-up outcomes. Distinguish tickets awaiting customer input from tickets awaiting internal action.

**Projected / hypothetical benefit:** More consistent follow-up may improve workload visibility and reduce tickets being overlooked. The dataset does not establish that this action will reduce the unresolved proportion.

### Priority 3 — Strengthen customer feedback capture and review

**Observed risk pattern:** Customer satisfaction feedback is available for only a subset of tickets.

**Evidence:** 2,769 of 8,469 tickets have recorded CSAT (32.70% coverage). The average recorded rating is 2.99/5; ratings of 1 or 2 account for 39.80% of recorded ratings.

**Operational action:** Review the process for requesting CSAT after ticket closure, monitor feedback coverage, and periodically review low-rated cases to identify service issues that warrant investigation.

**Projected / hypothetical benefit:** Better feedback coverage may provide management with a more complete view of customer experience and help target service reviews. The current dataset does not establish why customers gave low ratings or prove that this action will raise CSAT.

## 11. Limitations and responsible interpretation

1. **No explicit escalation indicator:** Actual escalated-ticket counts and escalation rates cannot be calculated.
2. **No queue or department field:** Queue-level comparisons are unavailable.
3. **No ticket-created timestamp:** First-response duration from ticket creation, total resolution duration from creation, and backlog age cannot be calculated.
4. **Timestamp ordering:** Negative intervals between first-response and resolution timestamps must be investigated and excluded from valid non-negative interval summaries.
5. **Missing CSAT:** Missing ratings are not zero and must not be assumed to indicate dissatisfaction.
6. **No SLA target:** SLA breach rates cannot be calculated.
7. **No causal claims:** Observed patterns are descriptive and do not prove what caused unresolved status or customer ratings.
8. **Dataset scope:** Conclusions apply to the supplied dataset and should not automatically be generalised to all customer-support operations.
9. **Privacy:** Do not publish customer names, email addresses, or raw ticket descriptions in reports, charts, or presentations.

## 12. Final conclusion

The analysis identifies cancellation requests as the ticket type with the highest unresolved proportion, a large overall unresolved workload, and limited customer-satisfaction feedback coverage. These observations support targeted operational review and improved feedback monitoring.

The three management actions above are **provisional evidence-based recommendations**. Before submission, the team should validate the corrected service-performance analysis, confirm all metric definitions and denominators, and ensure the final report uses the same results as the executed notebooks.